# Additional utilities

Small things around the methods: asking a language model, Wikipedia, posts from
Markdown, a tunnel for hooks.

In [6]:
import os

from amoorion import Amoorion

# A sandbox for this whole notebook: secrets only from variables, the inbox and
# the dataset in a temporary folder. Nothing here can touch your real setup.
Amoorion.init()

PosixPath('/tmp/amoorion-tfjfvzbz')

In [7]:
# A small fake of a language model, so nothing leaves this machine.
import json, threading

from http.server import BaseHTTPRequestHandler, HTTPServer


# It answers every request with the same short text.
class FakeModel(BaseHTTPRequestHandler):
    """Speaks the chat completions API, as Ollama or a LiteLLM proxy would."""

    def do_POST(self):

        asked = json.loads(self.rfile.read(int(self.headers["Content-Length"])))

        said = asked["messages"][-1]["content"].splitlines()[0][:50]

        answer = {
            "choices": [
                {"message": {"content": f"(a model's words about: {said})"}}
            ]
        }

        self.send_response(200)

        self.end_headers()

        self.wfile.write(json.dumps(answer).encode())

    def log_message(
        self,
        *args,
    ):

        pass


# Port 0: the system picks a free one.
model_server = HTTPServer(
    (
        "127.0.0.1",
        0,
    ),
    FakeModel,
)

# In the background, for the rest of the notebook.
threading.Thread(
    target=model_server.serve_forever,
    daemon=True,
).start()

# Where amoorion finds the model. Really these are `llm/url` and `llm/model` in your vault.
os.environ["AMOORION_SECRET_LLM_URL"] = (
    f"http://127.0.0.1:{model_server.server_port}/v1"
)

os.environ["AMOORION_SECRET_LLM_MODEL"] = "fake"

### `compose`: one call for many models

Ollama, llama.cpp's server, a LiteLLM proxy, OpenAI, Claude and Hugging Face's router
all speak this API, so `llm/url`, `llm/model` and `llm/token` in the vault choose one.
Each secret stays in the vault; a local server needs no token.

In [8]:
amo = Amoorion().use("dryrun")

# `system` says how to behave, `prompt` what to do. It writes; it posts nothing.
amo.compose(
    prompt="Announce our new release",
    system="Write one short sentence.",
)

{'text': "(a model's words about: Announce our new release)", 'model': 'fake'}

It chains with the rest in a workflow:

In [9]:
# Write, then publish: `$last` is the text of the previous step.
steps = [
    {
        "method": "compose",
        "params": {"prompt": "Announce our new release"},
    },
    {
        "method": "publish",
        "params": {
            "text": "$last",
            "to": ["dryrun"],
        },
    },
]


# The result of each step; [1] is publish.
amo.workflow(steps=steps)[1]["result"][0]["message"]["text"]

"(a model's words about: Announce our new release)"

### Posts from Markdown

Front matter gives the title, tags, links and platforms:

In [ ]:
from amoorion.utilities import post_from_markdown

# A post with its settings at the head of a Markdown file.
post, to = post_from_markdown(
    """---
title: Hello
tags: [news]
to: [dryrun]
---
Written once.
"""
)

print(
    post,
    to,
)


# The parsed post goes on to publish.
amo.publish(to=to, **post.model_dump())[0]["ok"]


# utilities... also known as helper functions
# most of simple pure python codes that handle things like processing of strings, hashing, encoding etc.
# should be functions among utilities

text='Written once.' title='Hello' links=[] tags=['news'] ['dryrun']


True

### `wiki`

A Wikipedia summary, as a message to quote or to feed a workflow. It needs the
network, so it is shown here and not run:

```python
amo.wiki(title="Blacksmith", lang="en")["text"]
```

### A tunnel for hooks

Hooks need an address reachable from outside. With `pip install ngrok` and
`ngrok/token` in the vault:

```python
from amoorion.utilities import open_tunnel

listener = open_tunnel(8000, amo.vault)
print(listener.url())   # give this to GitHub, plus /hooks/github
```

Anyone with that address reaches the port, so only the API belongs behind it: its
methods need their token and its hooks verify themselves.